# 07 · Spark y HDFS

Una cadena de tiendas quiere saber cuántas unidades ha vendido por zona. En esta práctica usamos tres pedidos. **HDFS guarda los datos y Spark calcula los totales.**

Necesitas el HDFS del curso en marcha y acceso desde el equipo donde ejecutas Spark. Consulta la [guía del entorno de PySpark](../01_environment_pyspark.md) si aún tienes que preparar Spark.

## Antes de empezar · Arrancar el Hadoop de la práctica anterior

En la **VM del curso**, abre una terminal en la carpeta `docker_hadoop` que clonaste en la práctica de Hadoop.

```bash
sudo docker-compose up -d
sudo docker ps
```

Comprueba que `namenode` y `datanode` estén en ejecución. Desde el navegador de la VM puedes abrir la [interfaz del NameNode](http://localhost:9870).

Abre este notebook con un **kernel en la misma VM**, esto es abrir Jupyter en el puerto 8888. Dentro del notebook, ``hdfs://localhost:9000`` apunta al NameNode de Hadoop. (El 8888 es para abrir Jupyter en el navegador y el 9000 es para que Spark se conecte a HDFS.)

La celda de abajo usa `hdfs://localhost:9000` para acceder a HDFS y guarda tres pedidos para comprobar la conexión.

Si aún tienes que instalar Docker o clonar el repositorio, sigue la [guía de preparación de Hadoop con Docker](../../Hadoop/00_examples/02_docker_environment_setup.md).

## 1. Preparar tres pedidos en HDFS

La ruta de abajo está preparada para ejecutar el notebook en la VM del curso. La celda guarda los pedidos en la subcarpeta `pedidos`. Si ya existe, cambia el nombre de la carpeta base antes de repetir la práctica.

In [ ]:
from pyspark.sql import SparkSession, functions as F

In [ ]:

base_hdfs = "hdfs://localhost:9000/user/root/practica_pedidos"
entrada_hdfs = f"{base_hdfs}/pedidos"
salida_hdfs = f"{base_hdfs}/resumen"

spark = SparkSession.builder.master("local[2]").appName("Pedidos en HDFS").getOrCreate()

pedidos = spark.createDataFrame(
    [("zona_3", 2), ("zona_1", 7), ("zona_3", 4)],
    ["zona", "unidades"],
)

pedidos.write.parquet(entrada_hdfs)

## 2. Leer los pedidos y sumar por zona

Spark lee los pedidos de HDFS. Agrupa las filas por `zona` y suma sus `unidades`. Al ejecutar `show()`, deberías ver **7** unidades para `zona_1` y **6** para `zona_3`.

In [ ]:
pedidos_hdfs = spark.read.parquet(entrada_hdfs)

resumen = pedidos_hdfs.groupBy("zona").agg(F.sum("unidades").alias("total_unidades"))
resumen.orderBy("zona").show()

## 3. Guardar el resumen en HDFS

Guardamos el resultado en la carpeta `resumen`. En la práctica, Spark ejecuta sus tareas en el equipo local. En un clúster, Spark puede repartir esas tareas entre varios equipos, mientras HDFS reparte los bloques de los archivos que guarda.

In [ ]:
resumen.write.parquet(salida_hdfs)
print(salida_hdfs)

## 4. Volver a consultar el resumen

Abre el resumen desde HDFS y comprueba los totales. También puedes ver las carpetas en la [interfaz del NameNode](http://localhost:9870), en **Utilities → Browse the file system**, dentro de `/user/root/practica_pedidos`.

In [ ]:
resumen_guardado = spark.read.parquet(salida_hdfs)
resumen_guardado.orderBy("zona").show()

In [ ]:
spark.stop()

Cerrar Jupyter o ejecutar spark.stop() no borra los datos guardados en HDFS. Solo termina la sesión que usaste para calcular el resumen.

Para consultarlo otro día, arranca Hadoop si está apagado, abre Jupyter en la misma VM y ejecuta la siguiente celda. Puedes ir directamente a la lectura, sin volver a crear los pedidos.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[2]").getOrCreate()

ruta = "hdfs://localhost:9000/user/root/practica_pedidos/resumen"
spark.read.parquet(ruta).orderBy("zona").show()

## Navegación

[Anterior · Parquet con pandas y PySpark](06_parquet_pandas_vs_pyspark.ipynb) · [Volver al README](../README.md) · [Siguiente · EDA y estadística con pandas](08_data_preparation_and_analysis.ipynb)